#  Feedback Aspect-Based Sentiment Analysis with Explainable AI of App Reviews


## Explainable AI Section


### Utilities

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
MODEL_DIR = "/content/drive/MyDrive/Sentiment-Project-Data/outputs/Models/distilroberta"
PREDS_CSV = "/content/drive/MyDrive/Sentiment-Project-Data/outputs/Preds/review_preds.csv"

from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval().to("cuda" if torch.cuda.is_available() else "cpu")

#### Stop-Word filter Customization

[The Full List of Negation and
Intensity words](https://aclanthology.org/attachments/P17-1154.Notes.pdf)

In [5]:
import spacy

nlp = spacy.load("en_core_web_sm")
stopwords = nlp.Defaults.stop_words

# The following list is from: https://aclanthology.org/attachments/P17-1154.Notes.pdf
# Removal of negation words
stopwords -= {"no", "not", "none", "never", "neither", "nobody",
"nothing", "nowhere", "seldom", "scarcely",
"hardly", "barely", "is not", "cannot", "may not",
"could not", "would not", "did not", "do not",
"does not", "was not", "are not", "were not"}

# Removal of intensity words
stopwords -= {"awfully", "extraordinary", "unusual", "much",
"rather", "very", "entirely", "greatly", "really",
"exceedingly", "too", "completely", "terribly",
"perfectly", "quite", "certainly", "especially",
"extremely", "fairly", "highly", "increasingly",
"much more", "particularly", "probably",
"more", "absolutely", "intensely", "supremely",
"most", "pretty"}

# The following list is from: https://factnow.wordpress.com/wp-content/uploads/2017/03/english-pronouns-list-pdf.pdf
# Adding pronouns to the stopwords
pronoun_stopwords = {
   "i", "you", "he", "she", "it", "we", "they", # Subject pronouns
    "me", "you", "him", "her", "it", "us", "them", #Object Pronouns
    "my", "your", "his", "her", "its", "our", "their", # Possesive adjectives
    "mine", "yours", "his", "hers", "its", "ours", "theirs", # Possessive pronouns
    "myself", "yourself", "himself", "herself", # Reflexive pronouns
    "itself", "ourselves", "yourselves", "themselves",
    "who", "whom", "whose", "which", "that",  # Relative pronouns
    "where", "when", "why", "what", "of which",
    "this", "that", "these", "those"  # Demonstrative pronouns
}
stopwords |= pronoun_stopwords

#print(stopwords)

def is_stopword(token_text: str) -> bool:
    """
    Return True if the entire fragment is essentially stopwords / junk.
    """
    t = text.lower().strip()
    if not t:
        return True
    words = [w for w in re.split(r"\s+", t) if w]
    # If all words are stopwords or 1-char junk, treat the whole fragment as stop
    if all((w in stopwords) or (len(w) == 1) for w in words):
        return True
    return False

In [6]:
import re, string, numpy as np
from typing import List, Tuple, Optional, Dict
import torch


def clean_span(text: str) -> str:
  """Trim punctuation and whitespace around extracted phrase"""
  return re.sub(r"^\W+|\W+$", "", text.strip())


def is_informative(token: str) -> bool:
    """Filter tokens using spaCy custom stopwords + punctuation + very short tokens."""
    t = token.lower().strip()
    if not t:
        return False
    if t in stopwords:
        return False
    if all(ch in string.punctuation for ch in t):
        return False
    if len(t) == 1:
      return False
    return True

def phrases_from_offsets(text: str, seq_ids: List[Optional[int]],
                         offsets: List[Tuple[int,int]],
                         scores: np.ndarray,
                         which_seq: int = 0,
                         topk: int = 6) -> List[str]:
  "Merge top-salient tokens (by scores) into readable phrases using offsets"
  idxs = [j for j,(sid,off) in enumerate(zip(seq_ids, offsets)) if sid==which_seq and off and off[1]>off[0]]
  if not idxs:
    return []
  svals = np.array([scores[j] for j in idxs])
  order = np.argsort(-svals)
  top_idxs = [idxs[i] for i in order[:max(topk,1)]]
  top_idxs.sort()

   #merge adjacent/nearby tokens
  spans, cur = [], None
  for j in top_idxs:
    start, end = offsets[j]
    if cur is None: cur = [start,end]
    elif start <= cur[1] + 1: cur[1] = max(cur[1], end)
    else: spans.append(tuple(cur)); cur = [start,end]
  if cur: spans.append(tuple(cur))

  out = []
  seen = set()
  for s,e in spans:
    frag = clean_span(text[s:e])
    if is_informative(frag):
        key = frag.lower()
        if key not in seen:
          seen.add(key); out.append(frag)
  if not out:
    for j in top_idxs:
      s,e = offsets[j]
      frag = clean_span(text[s:e])
      if frag: out.append(frag)
      if len(out) >= topk: break
  return out[:topk]


### Integrated Gradients Captum

In [ ]:
!pip install -q captum
!pip install -q tqdm

In [ ]:
import os, gc
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from captum.attr import IntegratedGradients

# ---------------- CONFIG ------------------------
INPUT_CSV = PREDS_CSV
OUTPUT_CSV = "/content/drive/MyDrive/Sentiment-Project-Data/outputs/review_preds_with_xai.csv"
TOKEN_SCORES_CSV = "/content/drive/MyDrive/Sentiment-Project-Data/outputs/review_token_scores_ig.csv"

BATCH = 4
MAX_LEN = 128
IG_STEPS = 16
TOPK = 6
USE_ASPECT_PAIR = True
ASPECT_FALLBACK = None

device = next(model.parameters()).device
model.to(device).eval()


df = pd.read_csv(INPUT_CSV)

print("Original rows:", len(df))

df = df[df["pred_label_str"].str.lower() != "absent"].copy()
df.reset_index(drop=True, inplace=True)

print("Rows after removing absent sentiments:", len(df))

text_col = "text" if "text" in df.columns else None
aspect_col = "aspect" if "aspect" in df.columns else None
if not text_col or not aspect_col:
  raise ValueError(f"Could not find text and aspect columns in {INPUT_CSV}; columns are {df.column.tolist()}")
texts = df[text_col].astype(str).tolist()
aspects = df[aspect_col].astype(str).tolist() if (aspect_col and USE_ASPECT_PAIR) else [None]*len(texts)

# Integrated Gradients Captum Setup
emb_layer = model.get_input_embeddings()
ig = IntegratedGradients(lambda emb, attn: model(inputs_embeds = emb,
                                                 attention_mask = attn,
                                                 return_dict = True).logits)
def tokenize_batch(t_list, a_list):
  enc = tokenizer(
      t_list,
      text_pair = a_list,
      padding = True,
      truncation = True,
      max_length = MAX_LEN,
      return_tensors = "pt",
      return_offsets_mapping = True
  )
  offsets = enc.pop("offset_mapping")
  encs = enc.encodings
  return enc, offsets, encs

def ig_batch(t_list, a_list):
    enc, offsets, encs = tokenize_batch(t_list, a_list)   # returns enc, offsets, encs
    ids = enc["input_ids"].to(device)
    attn = enc["attention_mask"].to(device)

    # forward for targets
    with torch.no_grad():
        logits = model(input_ids=ids, attention_mask=attn, return_dict=True).logits
        probs  = torch.softmax(logits, dim=-1)
        preds  = probs.argmax(dim=-1)

    # embeddings & baselines
    embeds = emb_layer(ids)
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
      pad_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0
    base_ids = torch.full_like(ids, pad_id)
    base_embeds = emb_layer(base_ids)

    # IG wrt predicted class
    atts = ig.attribute(
        inputs=embeds,
        baselines=base_embeds,
        additional_forward_args=(attn,),
        target=preds,
        n_steps=IG_STEPS
    )

    token_scores = atts.sum(dim=-1).detach().cpu().numpy()
    return token_scores, probs.cpu().numpy(), preds.cpu().numpy(), offsets.cpu().tolist(), encs

def extract_top_phrases(text, seq_ids, offsets, scores, topk=TOPK):
  """
    text:   original review string
    seq_ids: list[int|None] from fast tokenizer (0=text, 1=aspect, None=special)
    offsets: list[[start,end]] per token
    scores: 1D array-like per-token attribution scores for this example
    topk:   number of phrases to return for support/oppose
    Returns: (support_phrases, oppose_phrases)
  """
  idxs = [j for j, (sid, off) in enumerate(zip(seq_ids, offsets)) if sid == 0 and off and off[1] > off[0]]
  if not idxs:
    return [], []

  s = np.asarray(scores, dtype=float)
  svals = s[idxs] if s.ndim == 1 else np.array([scores[i] for i in idxs], dtype=float)
  pos_ord = np.argsort(-svals)[:max(1, topk)]
  neg_ord = np.argsort( svals)[:max(1, topk)]
  pos_token_ids = sorted([idxs[k] for k in pos_ord])
  neg_token_ids = sorted([idxs[k] for k in neg_ord])

  def merge(token_ids):
    spans, cur = [], None
    for j in token_ids:
      s0, e0 = offsets[j]
      if cur is None: cur = [s0, e0]
      elif s0 <= cur[1]+1: cur[1] = max(cur[1], e0)
      else: spans.append(tuple(cur)); cur = [s0, e0]
    if cur: spans.append(tuple(cur))

    out, seen = [], set()
    for s0, e0 in spans:
      frag = clean_span(text[s0:e0])
      if frag and not is_stopword(frag):
        key = frag.lower()
        if key not in seen:
          seen.add(key); out.append(frag)
    return out[:topk]

  support = merge(pos_token_ids)
  oppose  = merge(neg_token_ids)
  return support, oppose

# id2label mapping
ID2LABEL= {
    0 : "negative",
    1 : "neutral",
    2 : "positive",
    3:  "absent"
}

# Attribution loop
support_list, oppose_list, pred_labels, pred_probs = [], [], [], [], []
token_scores_all, token_row_idx = [], []

with tqdm(total = len(texts), desc = "Using IG Captum for Explaining") as pbar:
  for i0 in range(0, len(texts), BATCH):
        b_texts, b_aspects = texts[i0:i0+BATCH], aspects[i0:i0+BATCH]
        scores, probs, preds, offs, encs = ig_batch(b_texts, b_aspects)
        for bi, text in enumerate(b_texts):
            seq_ids = encs[bi].sequence_ids
            pos, neg = extract_top_phrases(text, seq_ids, offs[bi], scores[bi], topk=TOPK)
            pred_idx = int(preds[bi])
            if int(pred_idx) == 3:
              pred_labels.append("absent")
              pred_probs.append(probs[bi].tolist())
              support_list.append("")
              oppose_list.append("")
              continue
            pred_labels.append(ID2LABEL.get(pred_idx, str(pred_idx)))
            pred_probs.append(probs[bi].tolist())
            support_list.append("; ".join(pos))
            oppose_list.append("; ".join(neg))
            token_scores_all.append(scores[bi].tolist())
            token_scores_all.append(scores[bi].tolist())
            token_row_idx.append(i0 + bi)
        pbar.update(len(b_texts))
        del scores, probs, preds, offs, encs
        torch.cuda.empty_cache(); gc.collect()

# Merge and export
out_df = df.copy()
out_df["support"] = support_list
out_df["oppose"] = oppose_list
out_df["pred_label_xai"] = pred_labels
out_df["pred_probs_xai"] = pred_probs
out_df["evidence_support_topk"] = support_list
out_df["evidence_oppose_topk"] = oppose_list
out_df["explanation_extractive"] = out_df.apply(
    lambda r: f"Predicted {r['pred_label_xai']}. Supporting cues: {r['evidence_support_topk']}. Opposing cues: {r['evidence_oppose_topk']}.",
    axis=1
)

out_df.to_csv(OUTPUT_CSV, index=False)
print(f"Explanations saved to: {OUTPUT_CSV}")
print(f"[INFO] out_df shape: {out_df.shape}")

print("[INFO] token_scores_all length:", len(token_scores_all))
print("[INFO] token_row_idx length   :", len(token_row_idx))

token_df = pd.DataFrame({
    "row_idx": token_row_idx,
    "token_scores": token_scores_all
})


token_df.to_csv(TOKEN_SCORES_CSV, index=False)
print(f"[✓] Token scores saved to: {TOKEN_SCORES_CSV}")
print(f"[INFO] token_df shape: {token_df.shape}")

In [3]:
import pandas as pd

#5% sample
sample_df = pd.read_csv("/content/drive/MyDrive/Sentiment-Project-Data/outputs/review_preds_with_xai.csv")
sample_size = int(0.05*len(sample_df))
sample_df.sample(sample_size)


,app_id,app_name,store,id,text,aspect,pred_label_id,pred_label_str,prob,support,oppose,pred_label_xai,pred_probs_xai,evidence_support_topk,evidence_oppose_topk,explanation_extractive
36677,gov.irs,IRS2Go,google_play,c63ee20a-3c6f-46a3-8a4e-84ba367a12a7,I like it,general-satisfaction,2,positive,0.986369,I like it,I like it,positive,"[0.0024812892079353333, 0.0029784899670630693,...",I like it,I like it,Predicted: positive. Supporting cues: I like i...
32749,gov.irs,IRS2Go,google_play,fc98b9b2-d2b2-4ed1-90ed-95e7791de87f,Great App to check your refund status... easy ...,app-website,2,positive,0.989512,Great App; easy; great,your refund status; to; IRS,positive,"[0.00058823759900406, 0.0003816230164375156, 0...",Great App; easy; great,your refund status; to; IRS,Predicted: positive. Supporting cues: Great Ap...
31531,gov.irs,IRS2Go,google_play,f5262b93-92f9-47a7-bc67-2743098129be,Doesn't work after update.,app-website,0,negative,0.982849,Doesn't work after update,Doesn't work after,negative,"[0.9828487038612366, 0.005644347984343767, 0.0...",Doesn't work after update,Doesn't work after,Predicted: negative. Supporting cues: Doesn't ...
23275,gov.irs,IRS2Go,google_play,49c36012-f007-4c7f-b9ae-892d9bd5a799,Govermemt hard at work as usual,general-satisfaction,2,positive,0.827462,G; t; at work as usual,Govermemt hard at,positive,"[0.004845425486564636, 0.0025292118079960346, ...",G; t; at work as usual,Govermemt hard at,Predicted: positive. Supporting cues: G; t; at...
46250,1234298467,BC Services Card,app_store,11801807111,Signing into car is a nightmare…. No wonder so...,account-access,0,negative,0.923380,Signing into; is; nightmare; frustrated,car; wonder; talk; agent; MORE HELP,negative,"[0.9233801960945129, 0.003057193709537387, 0.0...",Signing into; is; nightmare; frustrated,car; wonder; talk; agent; MORE HELP,Predicted: negative. Supporting cues: Signing ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9970,1096545820,NaN,app_store,5426022431,Extremely happy of this free app!,app-website,2,positive,0.995512,Extremely happy; free app,Extremely; of this free,positive,"[0.0008785290410742164, 0.0006974023999646306,...",Extremely happy; free app,Extremely; of this free,Predicted: positive. Supporting cues: Extremel...
45294,gov.dhs.cbp.pspd.mpc,Mobile Passport Control,google_play,31c4193f-940c-4f30-8d6d-3499b10cc1ed,first time using...slow,speed,0,negative,0.681084,first time using...slow,first time using...slow,negative,"[0.6810867190361023, 0.010333292186260223, 0.1...",first time using...slow,first time using...slow,Predicted: negative. Supporting cues: first ti...
1807,ca.bc.gov.id.servicescard,BC Services Card,google_play,ddd7f439-dd91-4f33-8bf2-53fbfdc6911f,The new video call option is FANTASTIC for uno...,competitor,2,positive,0.868187,FANTAST; home; Best; ever; Thanks,un; forgot to; took; seconds; option,positive,"[0.0031301311682909727, 0.0004171253531239927,...",FANTAST; home; Best; ever; Thanks,un; forgot to; took; seconds; option,Predicted: positive. Supporting cues: FANTAST;...
41441,ca.gc.cbsa.coronavirus,ArriveCAN,google_play,bb958cb0-fb7c-44eb-9a5c-0b96e68591f8,Its user friendly App,app-website,2,positive,0.991225,Its user friendly App,Its user friendly App,positive,"[0.0014205747283995152, 0.0007306514889933169,...",Its user friendly App,Its user friendly App,Predicted: positive. Supporting cues: Its user...


### Evaluation Metrics

In [ ]:
!pip install vaderSentiment

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
model.to(device)

In [ ]:
import numpy as np
import pandas as pd
import torch
import ast
from sklearn.metrics import accuracy_score
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm.auto import tqdm


analyzer = SentimentIntensityAnalyzer()
df_preds = pd.read_csv("/content/drive/MyDrive/Sentiment-Project-Data/outputs/review_preds_with_xai.csv")
df_scores = pd.read_csv("/content/drive/MyDrive/Sentiment-Project-Data/outputs/review_token_scores_ig.csv")

# Merging token scores and predictions with XAI explanations
df_preds = df_preds.reset_index().rename(columns={"index": "row_idx"})
def parse_scores(x):
    try:
        return ast.literal_eval(x)
    except:
        return []
df_scores["token_scores"] = df_scores["token_scores"].apply(parse_scores)

df_eval = pd.merge(df_preds, df_scores, on="row_idx", how="inner")
print("Merged rows:", len(df_eval))


# (1) LABEL AGREEMENT
label_agreement = accuracy_score(df_eval["pred_label_str"], df_eval["pred_label_str"])
print("Label Agreement:", label_agreement)

# Helper function: polarity score
def polarity(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0
    return analyzer.polarity_scores(text)["compound"]

# Helper function: predict probabilities
def predict_probabilities(text, aspect):
    # Coerce text to string (handles NaN, floats, etc.)
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)

    # For aspect, if it's not a proper string, just ignore (no text_pair)
    if not isinstance(aspect, str) or not aspect.strip():
        aspect = None

    enc = tokenizer(
        text,
        text_pair=aspect,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()

    return probs

# Helper function: masking for metrics
def ig_mask_text(text, token_scores, keep_top_k=None, remove_top_k=None):
    """
    Build a masked version of 'text' by keeping or removing top-k tokens
    according to token_scores. Handles non-string / NaN text defensively.
    """
    # Coerce text to string
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    tokens = text.split()
    scores = np.array(token_scores)
    if len(tokens) == 0 or len(scores) == 0:
        return text
    L = min(len(tokens), len(scores))
    tokens = tokens[:L]
    scores = scores[:L]

    if keep_top_k is not None:
        keep_top_k = min(keep_top_k, L)
        idxs = np.argsort(scores)[::-1][:keep_top_k]
        kept = [tokens[i] for i in idxs]
        return " ".join(kept)

    if remove_top_k is not None:
        remove_top_k = min(remove_top_k, L)
        idxs = set(np.argsort(scores)[::-1][:remove_top_k])
        kept = [tokens[i] for i in range(L) if i not in idxs]
        return " ".join(kept)

    return text


K = 5  # top-k tokens to remove/keep
LABELS = ["negative", "neutral", "positive", "absent"]
LABEL2ID = {s: i for i, s in enumerate(LABELS)}

comprehensiveness_list = []
sufficiency_list = []
sparsity_list = []
local_sensitivity_list = []
monotonicity_list = []
support_polarity_list = []
oppose_polarity_list = []

print("Computing XAI metrics...")

for idx, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="XAI Metrics"):
    text = row.text
    aspect = row.aspect
    expl = np.array(row.token_scores)
    abs_expl = np.abs(expl)

    # LABEL probability
    pred_id = LABEL2ID[row.pred_label_str]
    original_prob = predict_probabilities(text, aspect)[pred_id]

    # COMPREHENSIVENESS
    # Remove most important tokens
    masked_text = ig_mask_text(text, abs_expl, remove_top_k=K)
    masked_prob = predict_probabilities(masked_text, aspect)[pred_id]
    comprehensiveness_list.append(original_prob - masked_prob)

    # SUFFICIENCY
    # Only most important tokens
    kept_text = ig_mask_text(text, abs_expl, keep_top_k=K)
    kept_prob = predict_probabilities(kept_text, aspect)[pred_id]
    sufficiency_list.append(original_prob - kept_prob)

    # SPARSITY
    sparsity_list.append(np.mean(abs_expl == 0))

    # LOCAL SENSITIVITY
    # Perturb explanation slightly
    eps = 1e-4
    perturbed = abs_expl + np.random.normal(0, eps, len(abs_expl))
    local_sensitivity_list.append(np.mean(np.abs(perturbed - abs_expl)))

    # MONOTONICITY
    # Increasing top-K tokens should not decrease prob
    monotonic_vals = []
    sorted_idx = np.argsort(abs_expl)[::-1]

    for k in range(1, min(6, len(sorted_idx))):
        top_text = ig_mask_text(text, abs_expl, keep_top_k=k)
        p = predict_probabilities(top_text, aspect)[pred_id]
        monotonic_vals.append(p)

    if len(monotonic_vals) > 1:
        monotonicity_list.append(np.mean(np.diff(monotonic_vals) >= 0))
    else:
        monotonicity_list.append(np.nan)

    # POLARITY ALIGNMENT
    support_polarity_list.append(polarity(row.support))
    oppose_polarity_list.append(polarity(row.oppose))


# Add metrics to df
df_eval["comprehensiveness"] = comprehensiveness_list
df_eval["sufficiency"]        = sufficiency_list
df_eval["sparsity"]           = sparsity_list
df_eval["local_sensitivity"]  = local_sensitivity_list
df_eval["monotonicity"]       = monotonicity_list
df_eval["support_polarity"]   = support_polarity_list
df_eval["oppose_polarity"]    = oppose_polarity_list

print("XAI metrics computed")

summary = df_eval[
    [
        "comprehensiveness",
        "sufficiency",
        "sparsity",
        "local_sensitivity",
        "monotonicity",
        "support_polarity",
        "oppose_polarity",
    ]
].mean()

print("\n------------- METRICS SUMMARY -------------")
print(summary)


Merged rows: 54150
Label Agreement: 1.0
Computing XAI metrics...


XAI Metrics:   0%|          | 0/54150 [00:00<?, ?it/s]

XAI metrics computed!

------------- METRICS SUMMARY -------------
comprehensiveness    0.339067
sufficiency          0.396375
sparsity             0.207711
local_sensitivity    0.000080
monotonicity         0.755416
support_polarity     0.178843
oppose_polarity      0.165421
dtype: float64


In [4]:
# Cleaning for gemini
import pandas as pd

df_eval = pd.read_csv("/content/drive/MyDrive/Sentiment-Project-Data/outputs/review_preds_with_xai.csv")
df_clean = df_eval.drop(columns=["pred_label_xai"], errors="ignore")
df_clean.head()

,app_id,app_name,store,id,text,aspect,pred_label_id,pred_label_str,prob,support,oppose,pred_probs_xai,evidence_support_topk,evidence_oppose_topk,explanation_extractive
0,gov.dhs.cbp.cbpone,CBP Link,google_play,e56d5632-cee9-46c2-8bb6-45c9b8aee608,"I was really enjoying this app, but I got tire...",app-website,2,positive,0.656943,really enjoying this app; enjoyed; good,but; tired; essentially; never; ethnic; violat...,"[0.2506444454193115, 0.0193485040217638, 0.726...",really enjoying this app; enjoyed; good,but; tired; essentially; never; ethnic; violat...,Predicted: positive. Supporting cues: really e...
1,gov.dhs.cbp.cbpone,CBP Link,google_play,34a0feb3-57f0-4dce-b3d0-8120251b1676,Folks said people get 1000 dollars if they sel...,general-satisfaction,2,positive,0.932401,ol; This is very good,people; if; dep; they; legally,"[0.02376604452729225, 0.002903377404436469, 0....",ol; This is very good,people; if; dep; they; legally,Predicted: positive. Supporting cues: ol; This...
2,gov.dhs.cbp.cbpone,CBP Link,google_play,5af55159-71da-4ef4-8bc8-9fc4ca1f41c6,Update: I'm still in America. No one's given m...,account-access,0,negative,0.958933,scam; t; me log in,m; home; my home,"[0.9589329361915588, 0.004514071624726057, 0.0...",scam; t; me log in,m; home; my home,Predicted: negative. Supporting cues: scam; t;...
3,gov.dhs.cbp.cbpone,CBP Link,google_play,fe8039ce-34bc-4d1b-8eff-5097f55ca9de,"The application on iOS, when reading the passp...",app-website,0,negative,0.953230,The; iOS; error; not; applications,reading; and; However,"[0.9639237523078918, 0.019808514043688774, 0.0...",The; iOS; error; not; applications,reading; and; However,Predicted: negative. Supporting cues: The; iOS...
4,gov.dhs.cbp.cbpone,CBP Link,google_play,1668b3f8-9ec5-4d43-afa9-f45183ab93fc,"It doesn't allow me to take photo, an error me...",app-website,0,negative,0.744965,It doesn't; error; saying,photo; my face and; believe; assist,"[0.6862407922744751, 0.0015585338696837425, 0....",It doesn't; error; saying,photo; my face and; believe; assist,Predicted: negative. Supporting cues: It doesn...


## Gemini Justification

In [ ]:
!pip install -U google-generativeai

In [6]:
import os
import google.generativeai as genai
from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

In [7]:
from textwrap import shorten
def build_gemini_prompt(row) -> str:
  """
  Uses only the cues from IG + predicted label
  """

  text   = shorten(str(row["text"]), width=600, placeholder=" ...")
  aspect = str(row["aspect"]) if not pd.isna(row["aspect"]) else None
  label  = str(row.get("pred_label_str", row.get("pred_label_xai", row["pred_label_id"])))
  sup    = str(row.get("evidence_support_topk", "")).strip()
  opp    = str(row.get("evidence_oppose_topk", "")).strip()

  aspect_part = f" for the aspect **{aspect}**" if aspect else ""
  aspect_part = f" for the aspect **{aspect}**" if aspect else ""
  support_part = sup if sup else "N/A"
  oppose_part  = opp if opp else "N/A"

  prompt = f"""
  You are an explainable AI assistant.

  A sentiment model predicted **{label}** sentiment{aspect_part} for the following user review of an app.
  The review text and the key evidence phrases (from an attribution method) are provided.

  Your job:
  - Write a short justification (2–4 sentences, ≤80 words).
  - Base your reasoning **only** on the evidence phrases given.
  - Use the opposing cues (if any) to mention nuances or mixed signals.
  - Do NOT invent new evidence or speculate about user intent beyond what is shown.

  Review text:
  \"\"\"{text}\"\"\"

  Supporting cues (evidence for the predicted sentiment):
  {support_part}

  Opposing cues (evidence against the predicted sentiment):
  {oppose_part}

  Now write the justification in plain English, as if explaining the model's decision to a product team.
  """
  return prompt.strip()


def generate_gemini_justification(row) -> str:
  prompt = build_gemini_prompt(row)
  print(f"\n[STEP] Calling Gemini for row {row.name}")
  try:
    resp = gemini_model.generate_content(prompt)
    print("Got response.")
    return (resp.text or "").strip()
  except Exception as e:
    print(f"Gemini error on row {row.name}: {repr(e)}")

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

df_gemini = df_clean

MAX_PER_GROUP = 10
MAX_ROWS_GLOBAL = 100
group_cols = ["app_id", "aspect", "pred_label_str"]

groups = df_gemini.groupby(group_cols)

samples = []
for _, g in groups:
    samples.append(g.sample(min(len(g), MAX_PER_GROUP), random_state=58))

sampled = pd.concat(samples, ignore_index=True)

if len(sampled) > MAX_ROWS_GLOBAL:
    sampled = sampled.sample(MAX_ROWS_GLOBAL, random_state=58)

rows = sampled.reset_index(drop=True)
print("Rows to send to Gemini:", len(rows))

justifications = []
for _, row in tqdm(rows.iterrows(), total=len(rows), desc="Gemini justifications"):
    justifications.append(generate_gemini_justification(row))

if "gemini_justification" not in df_gemini.columns:
    df_gemini["gemini_justification"] = ""

df_gemini.loc[rows.index, "gemini_justification"] = justifications
df_gemini_sampled = df_gemini.loc[rows.index].copy()

OUTPUT_CSV_GEMINI = "/content/drive/MyDrive/Sentiment-Project-Data/outputs/review_preds_with_xai_and_gemini_sample.csv"
df_gemini_sampled.to_csv(OUTPUT_CSV_GEMINI, index=False)

print(f"Saved Gemini-sampled rows ({len(df_gemini_sampled)}) to: {OUTPUT_CSV_GEMINI}")